In [0]:
print("Singapore Vessel Traffic Pipeline")
print("Environment is ready!")

Singapore Vessel Traffic Pipeline
Environment is ready!


In [0]:
file_path = "/Volumes/workspace/vessel_arrival_data_pipeline/vesselarrivals/Vessel Arrivals (!75 GT) Breakdown, Monthly.csv"

df = spark.read \
    .option("header", True) \
    .option("inferschema", True) \
    .csv(file_path)

display(df)

month,vessel_type,number_of_vessels,gross_tonnage
1995-01-01,Container,1112,19477.72
1995-01-01,Freighter,916,4006.11
1995-01-01,Bulk Carrier,346,9115.43
1995-01-01,Tanker,1272,20064.49
1995-01-01,Passenger,2656,2403.33
1995-01-01,Barges,1352,1241.02
1995-01-01,Tug,775,129.44
1995-01-01,Miscellaneous,447,2541.08
1995-02-01,Container,1001,18105.27
1995-02-01,Freighter,843,3995.91


In [0]:
#Check column names and data types
df.printSchema()

#Check how many rows available
print("Total Rows:", df.count())

#Preview first 5 rows
display(df.limit(5))

#Check for missing values in each column 
from pyspark.sql import functions as F

df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

root
 |-- month: date (nullable = true)
 |-- vessel_type: string (nullable = true)
 |-- number_of_vessels: integer (nullable = true)
 |-- gross_tonnage: double (nullable = true)

Total Rows: 3048


month,vessel_type,number_of_vessels,gross_tonnage
1995-01-01,Container,1112,19477.72
1995-01-01,Freighter,916,4006.11
1995-01-01,Bulk Carrier,346,9115.43
1995-01-01,Tanker,1272,20064.49
1995-01-01,Passenger,2656,2403.33


+-----+-----------+-----------------+-------------+
|month|vessel_type|number_of_vessels|gross_tonnage|
+-----+-----------+-----------------+-------------+
|    0|          0|                0|            0|
+-----+-----------+-----------------+-------------+



In [0]:
#Count exact duplicate rows
total_rows = df.count()
distinct_rows = df.distinct().count()

print("Total Rows:", total_rows)
print("Distinct Rows:", distinct_rows)
print("Duplicate Rows:", total_rows - distinct_rows)

Total Rows: 3048
Distinct Rows: 3048
Duplicate Rows: 0


In [0]:
file_path = "/Volumes/workspace/vessel_arrival_data_pipeline/vesselarrivals/Vessel Arrivals (!75 GT) Breakdown, Monthly.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

print("Total rows:", df.count())
df.printSchema()

Total rows: 3048
root
 |-- month: date (nullable = true)
 |-- vessel_type: string (nullable = true)
 |-- number_of_vessels: integer (nullable = true)
 |-- gross_tonnage: double (nullable = true)



In [0]:
from pyspark.sql import functions as F

duplicate_groups = (
    df.groupBy("month", "vessel_type")
    .count()
    .filter(F.col("count") > 1)
)

print("Repeated month-vessel type combinations:",
      duplicate_groups.count())

display(duplicate_groups)

Repeated month-vessel type combinations: 0


month,vessel_type,count


In [0]:
bronze_table = ("workspace.vessel_arrival_data_pipeline.bronze_vessel_arrivals"
                )

( df.write 
  .format("delta")
  .mode("overwrite")
  .saveAsTable(bronze_table)
  )

print("Bronze table created:", bronze_table)

Bronze table created: workspace.vessel_arrival_data_pipeline.bronze_vessel_arrivals


In [0]:
bronze_df = spark.table (
    "workspace.vessel_arrival_data_pipeline.bronze_vessel_arrivals"
)

print("Bronze row count:", bronze_df.count())
display(bronze_df.limit(5))

Bronze row count: 3048


month,vessel_type,number_of_vessels,gross_tonnage
1995-01-01,Container,1112,19477.72
1995-01-01,Freighter,916,4006.11
1995-01-01,Bulk Carrier,346,9115.43
1995-01-01,Tanker,1272,20064.49
1995-01-01,Passenger,2656,2403.33


In [0]:
from pyspark.sql.functions import col

silver_table = "workspace.vessel_arrival_data_pipeline.silver_vessel_arrivals"

silver_df = (
    bronze_df
    .filter(col("month").isNotNull())
    .filter(col("vessel_type").isNotNull())
    .filter(col("number_of_vessels").isNotNull())
    .filter(col("gross_tonnage").isNotNull())
    .filter(col("number_of_vessels") >= 0)
    .filter(col("gross_tonnage") >= 0)
    .dropDuplicates()
)


print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_df.count())

Bronze row count: 3048
Silver row count: 3048


In [0]:
silver_table = "workspace.vessel_arrival_data_pipeline.silver_vessel_arrivals"

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print("Silver table created:", silver_table)

Silver table created: workspace.vessel_arrival_data_pipeline.silver_vessel_arrivals


In [0]:
from pyspark.sql.functions import sum as spark_sum

gold_monthly_table = (
    "workspace.vessel_arrival_data_pipeline.gold_monthly_vessel_traffic"
)

gold_monthly_df = (
    silver_df
    .groupBy("month")
    .agg(
        spark_sum("number_of_vessels").alias("total_vessels"),
        spark_sum("gross_tonnage").alias("total_gross_tonnage")
    )
    .orderBy("month")
)

display(gold_monthly_df)

month,total_vessels,total_gross_tonnage
1995-01-01,8876,58978.62
1995-02-01,7593,53786.759999999995
1995-03-01,8610,58440.75000000001
1995-04-01,8483,59351.66
1995-05-01,8749,61696.060000000005
1995-06-01,8507,55360.23999999999
1995-07-01,8669,61310.53
1995-08-01,8498,60099.24
1995-09-01,8582,58360.89000000001
1995-10-01,9140,60323.65


In [0]:
(
    gold_monthly_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_monthly_table)
)

print("Gold table created:", gold_monthly_table)

Gold table created: workspace.vessel_arrival_data_pipeline.gold_monthly_vessel_traffic


In [0]:
gold_check = spark.table(
    "workspace.vessel_arrival_data_pipeline.gold_monthly_vessel_traffic"
)

print("Gold row count:", gold_check.count())

display(gold_check.orderBy("month"))

Gold row count: 381


month,total_vessels,total_gross_tonnage
1995-01-01,8876,58978.62
1995-02-01,7593,53786.759999999995
1995-03-01,8610,58440.74999999999
1995-04-01,8483,59351.66
1995-05-01,8749,61696.06
1995-06-01,8507,55360.24
1995-07-01,8669,61310.530000000006
1995-08-01,8498,60099.24
1995-09-01,8582,58360.89
1995-10-01,9140,60323.65000000001


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import lag, col, round as spark_round

monthly_window = Window.orderBy("month")

gold_trends_df = (
    gold_check
    .withColumn(
        "previous_month_vessels",
        lag("total_vessels").over(monthly_window)
    )
    .withColumn(
        "vessel_change",
        col("total_vessels") - col("previous_month_vessels")
    )
    .withColumn(
        "vessel_change_pct",
        spark_round(
            (
                col("vessel_change")
                / col("previous_month_vessels")
            ) * 100,
            2
        )
    )
)

display(gold_trends_df.orderBy("month"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


month,total_vessels,total_gross_tonnage,previous_month_vessels,vessel_change,vessel_change_pct
1995-01-01,8876,58978.62,null,null,null
1995-02-01,7593,53786.759999999995,8876,-1283,-14.45
1995-03-01,8610,58440.74999999999,7593,1017,13.39
1995-04-01,8483,59351.66,8610,-127,-1.48
1995-05-01,8749,61696.06,8483,266,3.14
1995-06-01,8507,55360.24,8749,-242,-2.77
1995-07-01,8669,61310.530000000006,8507,162,1.9
1995-08-01,8498,60099.24,8669,-171,-1.97
1995-09-01,8582,58360.89,8498,84,0.99
1995-10-01,9140,60323.65000000001,8582,558,6.5


In [0]:
gold_trends_table = (
    "workspace.vessel_arrival_data_pipeline.gold_vessel_traffic_trends"
)

(
    gold_trends_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_trends_table)
)

print("Gold trends table created:", gold_trends_table)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold trends table created: workspace.vessel_arrival_data_pipeline.gold_vessel_traffic_trends


In [0]:
%sql
SELECT
    month,
    total_vessels,
    previous_month_vessels,
    vessel_change,
    vessel_change_pct
FROM workspace.vessel_arrival_data_pipeline.gold_vessel_traffic_trends
ORDER BY month DESC
LIMIT 12;

month,total_vessels,previous_month_vessels,vessel_change,vessel_change_pct
2026-09-01,11608,11631,-23,-0.2
2026-08-01,11631,11655,-24,-0.21
2026-07-01,11655,11520,135,1.17
2026-06-01,11520,11729,-209,-1.78
2026-05-01,11729,10873,856,7.87
2026-04-01,10873,11591,-718,-6.19
2026-03-01,11591,10886,705,6.48
2026-02-01,10886,12031,-1145,-9.52
2026-01-01,12031,11892,139,1.17
2025-12-01,11892,11207,685,6.11


In [0]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round

vessel_type_df = (
    silver_df
    .groupBy("vessel_type")
    .agg(
        spark_sum("number_of_vessels").alias("total_vessels"),
        spark_sum("gross_tonnage").alias("total_gross_tonnage")
    )
    .orderBy("total_vessels", ascending=False)
)

display(vessel_type_df)

vessel_type,total_vessels,total_gross_tonnage
Passenger,1213887,1128769.26
Tanker,657918,1.9340892840000007E7
Container,540954,1.9318625949999996E7
Barges,409903,568315.7200000002
Tug,377863,77571.65000000005
Bulk Carrier,355285,1.6372507580000004E7
Miscellaneous,292257,3158599.7700000005
Freighter,263205,1321168.7299999997


In [0]:
gold_vessel_type_table = (
    "workspace.vessel_arrival_data_pipeline.gold_vessel_type_summary"
)

(
    vessel_type_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_vessel_type_table)
)

print("Gold vessel type table created:", gold_vessel_type_table)

Gold vessel type table created: workspace.vessel_arrival_data_pipeline.gold_vessel_type_summary


In [0]:
%sql
SHOW TABLES IN workspace.vessel_arrival_data_pipeline;

database,tableName,isTemporary
vessel_arrival_data_pipeline,bronze_vessel_arrivals,false
vessel_arrival_data_pipeline,gold_monthly_vessel_traffic,false
vessel_arrival_data_pipeline,gold_vessel_traffic_trends,false
vessel_arrival_data_pipeline,gold_vessel_type_summary,false
vessel_arrival_data_pipeline,silver_vessel_arrivals,false


In [0]:
%sql
SELECT *
FROM workspace.vessel_arrival_data_pipeline.gold_monthly_vessel_traffic
LIMIT 10;

month,total_vessels,total_gross_tonnage
1995-01-01,8876,58978.62
1995-02-01,7593,53786.759999999995
1995-03-01,8610,58440.74999999999
1995-04-01,8483,59351.66
1995-05-01,8749,61696.06
1995-06-01,8507,55360.24
1995-07-01,8669,61310.530000000006
1995-08-01,8498,60099.24
1995-09-01,8582,58360.89
1995-10-01,9140,60323.65000000001
